# 비언어 심볼
- `[laughter]`·`[sigh]`·`[surprise-oh]` 같은 감정 표현
- OmniVoice 는 텍스트 안에 대괄호 태그를 넣어 **웃음·한숨·놀람·동의** 같은 비언어 소리를 합성합니다. NPC 대사·내레이션·콘텐츠 더빙에서 감정 표현이 살아납니다.

> 지원 태그는 영어권 학습 데이터 기반이라 **영어 문장에 가장 잘 동작**. 한국어 문장 안에 섞어 써도 어느 정도 동작하지만 영어보다 안정도가 떨어질 수 있음.

## 0. 셋업

In [ ]:
# 라이브러리 다운로드
%pip install omnivoice

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.0/163.0 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 9.4 MB/s eta 0:00:00


- Hugging Face 토큰 에러 발생시 아래 코드 실행

In [ ]:
# import os

# os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
# try:
#     import huggingface_hub.constants as hf_constants
#     hf_constants.HF_HUB_DISABLE_IMPLICIT_TOKEN = True
# except Exception:
#     pass

In [2]:
import torch
import soundfile as sf
from omnivoice import OmniVoice
from IPython.display import Audio
import os
os.makedirs("outputs", exist_ok=True)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=device,
    dtype=dtype,
)
print("준비 완료. device:", device)

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

준비 완료. device: cuda:0


## 1. 지원하는 비언어 심볼

총 13가지. 텍스트 안 어디든 `[태그]` 형태로 끼워 넣으면 됨.

| 태그 | 의미 |
|---|---|
| `[laughter]` | 웃음 |
| `[sigh]` | 한숨 |
| `[confirmation-en]` | 동의·맞장구 (Mm-hmm 류) |
| `[question-en]`, `[question-ah]`, `[question-oh]`, `[question-ei]`, `[question-yi]` | 의문 (다섯 가지 톤) |
| `[surprise-ah]`, `[surprise-oh]`, `[surprise-wa]`, `[surprise-yo]` | 놀람 (네 가지 톤) |
| `[dissatisfaction-hnn]` | 불만 (Hmph 류) |

## 2. 단일 심볼, 웃음

In [5]:
audio = model.generate(
    text="[laughter] You really got me. I didn't see that coming at all."
)
sf.write("outputs/nv_laughter.wav", audio[0], 24000)
Audio("outputs/nv_laughter.wav")

## 3. 한숨, 의문, 놀람 비교

In [6]:
samples = {
    "nv_sigh"        : "[sigh] I guess we have to do this all over again.",
    "nv_question_ah" : "Wait. [question-ah] Is that really what you meant?",
    "nv_surprise_oh" : "[surprise-oh] No way! That actually worked?",
    "nv_dissatis"    : "[dissatisfaction-hnn] Fine. I'll do it your way then.",
}

for name, text in samples.items():
    audio = model.generate(text=text)
    path = f"outputs/{name}.wav"
    sf.write(path, audio[0], 24000)
    print(f"  saved: {path}")

  saved: outputs/nv_sigh.wav
  saved: outputs/nv_question_ah.wav
  saved: outputs/nv_surprise_oh.wav
  saved: outputs/nv_dissatis.wav


In [7]:
for name in samples:
    print(name)
    display(Audio(f"outputs/{name}.wav"))

nv_sigh


nv_question_ah


nv_surprise_oh


nv_dissatis


## 4. 한 문장 안에 여러 심볼

NPC 대사처럼 한 줄 안에 감정 흐름을 만들 수 있습니다.

In [9]:
text = "[sigh] So you finally came. [laughter] I knew you couldn't resist. [surprise-oh] But what's that in your hand?"
audio = model.generate(text=text)
sf.write("outputs/nv_combo.wav", audio[0], 24000)
Audio("outputs/nv_combo.wav")

## 5. Voice Design 과 조합, 캐릭터 + 감정

`instruct` 로 캐릭터 톤 고정 + 텍스트 안에 비언어 심볼. 시나리오 별로 가장 잘 먹히는 조합.

In [10]:
# 늙은 마법사가 웃으면서
audio = model.generate(
    text="[laughter] So, young one, you've come for the ancient secret? [sigh] Very well.",
    instruct="male, elderly, low pitch",
)
sf.write("outputs/nv_wizard_laugh.wav", audio[0], 24000)

# 발랄한 친구가 놀라며
audio = model.generate(
    text="[surprise-wa] No way you did that! [laughter] That's amazing!",
    instruct="female, young adult, high pitch",
)
sf.write("outputs/nv_friend_surprise.wav", audio[0], 24000)

# 보스가 불만스럽게
audio = model.generate(
    text="[dissatisfaction-hnn] You think you can defeat me with that? [laughter] Pathetic.",
    instruct="male, very low pitch",
)
sf.write("outputs/nv_boss_dissatis.wav", audio[0], 24000)

for name in ["nv_wizard_laugh", "nv_friend_surprise", "nv_boss_dissatis"]:
    print(name)
    display(Audio(f"outputs/{name}.wav"))

nv_wizard_laugh


nv_friend_surprise


nv_boss_dissatis


## 6. 한국어 텍스트 + 영어 비언어 심볼

심볼 자체는 영어 학습 데이터 기반이지만, 한국어 문장 안에서도 흐름 정도는 살아납니다. 안정도는 영어보다 낮음.

In [13]:
text = "[laughter] 정말이야? 그건 생각도 못 했어. [sigh] 태식아 왜 그러니 정말."
audio = model.generate(text=text)
sf.write("outputs/nv_korean_mix.wav", audio[0], 24000)
Audio("outputs/nv_korean_mix.wav")

## 7. 팁

- 심볼은 **문장 시작 또는 구절 사이** 에 놓는 게 가장 자연스러움
- 한 문장에 심볼 **2~3개 이내** 가 안정. 너무 많이 넣으면 오히려 부자연
- Voice Design (`instruct`) 또는 Voice Cloning (`ref_audio`) 와 조합 가능
- 영어가 가장 안정. 한국어는 실험적

## [실습]
1. 자신의 게임 캐릭터에 어울리는 감정 (웃음 / 한숨 / 놀람) 으로 영어 한 문장씩.
2. 같은 텍스트에 심볼 위치를 앞 / 중간 / 끝 으로 옮겨가며 결과 비교.
3. Voice Cloning (내 목소리) + 비언어 심볼 조합도 가능한지 시도.
4. 짧은 NPC 대화 시나리오 (3~4 문장) 에 비언어 심볼로 감정 곡선 만들기.